# Session 2: From Static Weights to Adaptive Allocation

In Session 1, we built a classical minimum-variance portfolio and watched it fail under stress. The core problem? Static weights can't respond to a changing world. In this session, we take humans _partially_ out of the loop: we design an AI rebalancing engine that uses **utility maximization** to allocate capital, reads market sentiment to adjust risk preferences dynamically, and enforces explicit safety rules.

The [Session 2 introduction notebook](eCornell-AI-Finance-S2-Introduction-MayasAdaptiveEngine-May-2026.ipynb) frames the same material as a Lindenfield scenario: Maya Chen's Session 1 Balanced book rode a twenty-one percent drawdown through August, the account is on the risk committee's agenda next Thursday, and Dr. Amara Okafor's utility-based framework is the candidate fix. Amara's pitch reduces to three operational claims: a signal, a utility function, and a set of trigger rules. Those claims are the three deliverables of this lecture, in order.

> __Learning Objectives:__
>
> * __Cobb-Douglas Utility Maximization:__ Formulate portfolio allocation as a budget-constrained Cobb-Douglas utility problem and derive the closed-form analytical solution. Compare the allocation behavior of Cobb-Douglas to CES and log-linear alternatives under the same preference weights.
> * __Sentiment-Driven Preference Weights:__ Construct a sentiment signal from EMA crossover and map it to a risk-aversion parameter that modulates preference weights. Feed the signal into SIM-based preference weights so the allocator adapts dynamically to changing market conditions.
> * __Rebalancing Engine with Safety Rules:__ Implement a daily rebalancing loop that computes allocations via utility maximization and enforces trigger rules. Configure drawdown limits, turnover caps, and reallocation schedules that encode human safety judgment into the engine.

Let's build a machine that adapts.

___

## Examples

The example notebooks below accompany this lecture. They split into __core__ examples that you should run live alongside the lecture, and __optional__ examples that extend the core material and can be studied afterward at your own pace.

### Core (tonight)

These four notebooks form the session's deliverable chain: build the allocator, wire it into an engine, stress-test the engine across 5,000 futures, and compare adaptive vs static path-by-path.

> __Run live with the lecture:__
>
> * [▶ Let's build the Cobb-Douglas allocator](eCornell-AI-Finance-S2-Example-BuildCobbDouglasAllocator-May-2026.ipynb). Generate a synthetic market path, compute SIM parameters and the EMA-crossover sentiment signal, build the Cobb-Douglas utility allocator, and compare its allocation behavior to CES and log-linear alternatives.
> * [▶ Let's wire the allocator into a rebalancing engine and produce a scorecard](eCornell-AI-Finance-S2-Example-RebalancingEngineScorecard-May-2026.ipynb). Run the Cobb-Douglas engine with trigger rules on a synthetic market path, compare it head-to-head against the Session 1 min-var and equal-weight baselines, and sweep the CES elasticity parameter to see how concentration tunes engine behavior.
> * [▶ Let's stress-test the rebalancing engine on 5,000 futures](eCornell-AI-Finance-S2-Example-StressTestRebalancingEngine-May-2026.ipynb). Generate the full Monte Carlo ensemble, run seven strategies, and compute the tail-risk scorecard (VaR, CVaR, drawdown, NPV). Produces the `stress-test-engine.jld2` hand-off that the remaining notebooks consume.
> * [▶ Let's compare static and adaptive allocation across 5,000 paths](eCornell-AI-Finance-S2-Example-StaticVsAdaptiveComparison-May-2026.ipynb). Run static and adaptive strategies head-to-head, measure win rates, identify regime dependence, and compare paired quantiles across the wealth distribution.
>
> Run all four in order. Notebook 3 produces the hand-off file that notebook 4 and both optional notebooks consume.

Once you complete the core examples, you will have a calibrated Cobb-Douglas rebalancing engine, a 5,000-path stress-test scorecard, and a paired per-path comparison against the Session 1 baseline that Session 3's learning extensions will build on.

### Optional (self-study)

These two notebooks extend the core material. One swaps the EMA-crossover sentiment signal for the HMM regime posterior; the other diagnoses trading cost, attribution, and rebalance dynamics. Neither is required to reach the Session 3 hand-off.

> __Extension material (self-study):__
>
> * [▶ Let's replace EMA crossover with the HMM regime posterior](eCornell-AI-Finance-S2-Example-RegimeAwareSentiment-May-2026.ipynb). Build a regime-aware $\lambda$ signal from the HMM posterior, run the engine with both EMA and regime-based sentiment on the same path, and compare their distributional performance across 5,000 synthetic futures.
> * [▶ Let's diagnose turnover cost and attribution](eCornell-AI-Finance-S2-Example-TurnoverAttributionDiagnostics-May-2026.ipynb). Measure transaction cost sensitivity, decompose performance via a 3-way attribution (utility choice, dynamic reallocation, trigger rules), and visualize the rebalance-event timeline.
>
> Study either when you want to understand where the core engine's sentiment signal or performance edge really comes from.

___

## Why Static Weights Fail

Session 1 showed us that the minimum-variance optimizer treats its inputs, the expected growth-rate vector $\mathbb{E}[\mathbf{g}]$ and the covariance matrix $\boldsymbol{\Sigma}_g$, as fixed constants. But markets are not stationary. Correlations spike during crises, volatilities cluster, and expected growth rates shift with economic regimes. A portfolio computed from last year's data has no mechanism to react when the world changes _today_.

> __The core limitation:__
>
> Static portfolio construction is a _one-shot_ decision: estimate, optimize, deploy. There is no feedback loop. No matter what happens after deployment, the weights don't move until a human intervenes. This is not a flaw in the math; it is a flaw in the _architecture_.

The fix is straightforward in concept: replace the one-shot decision with a _loop_ that continuously reads market conditions and adjusts the portfolio accordingly. But we need a principled framework for making those allocation decisions. That framework is **utility maximization**.

## Utility Maximization: A Framework for Allocation

Instead of asking "what portfolio minimizes variance?" we ask a different question: **given a budget and current market conditions, what portfolio maximizes the investor's utility?**

This shift is fundamental. Minimum-variance optimization treats all assets symmetrically; the only inputs are expected growth rates and covariances. Utility maximization lets us encode _preferences_: which assets do we like right now? How strongly? And those preferences can change every day based on market signals.

To connect Session 2 back to Session 1, at rebalance time $t$ we let $S_i(t)$ denote the price of asset $i$, $n_i(t)$ the shares held, $W_{\mathcal{P}}(t)$ the total portfolio wealth, and $w_i(t) = n_i(t)S_i(t)/W_{\mathcal{P}}(t)$ the portfolio weight. In the one-step allocation problem below, we abbreviate the current price by $P_i \equiv S_i(t)$ and the available budget by $B \equiv W_{\mathcal{P}}(t)$.

> __The general problem:__
>
> Given $N$ assets with current prices $P_i \equiv S_i(t)$ for $i=1,\ldots,N$ and a total budget $B \equiv W_{\mathcal{P}}(t)$, choose the number of shares $n_1, \ldots, n_N$ to:
>
> $$\max_{n_1, \ldots, n_N} \quad U(n_1, \ldots, n_N) \qquad \text{subject to} \quad \sum_{i=1}^{N} n_i \cdot P_i = B$$
>
> The utility function $U$ encodes the investor's preferences. Different choices of $U$ lead to different allocation strategies.

The key insight is that the preference weights $\gamma_i$ inside $U$ are not fixed; they are computed _dynamically_ from market sentiment and fundamentals. This is where the AI enters: it translates real-time signals into preference weights that drive the allocation.

### Cobb-Douglas Utility
The **Cobb-Douglas utility function** is the workhorse of this course. It has a clean analytical solution, encodes preferences naturally through exponents, and separates assets into "preferred" and "non-preferred" categories.

> __Cobb-Douglas Utility:__
>
> $$U(n_1, \ldots, n_N) = \kappa(\boldsymbol{\gamma}) \prod_{i=1}^{N} n_i^{\gamma_i}$$
>
> where $\gamma_i \in (-1, 1)$ is the preference exponent for asset $i$, and $\kappa = +1$ if all $\gamma_i \geq 0$, else $\kappa = -1$. The preference exponents do all the work:
> * $\gamma_i > 0$, **preferred**: the investor wants more of this asset. Higher $\gamma_i$ means stronger preference.
> * $\gamma_i \leq 0$, **non-preferred**: the investor wants to minimize exposure. These assets receive only a minimum position $\epsilon$.
>
> The Cobb-Douglas utility is a product of powers, which means the optimal allocation will be proportional to the preference weights raised to those powers. The budget constraint ensures we can't just buy infinite shares of the preferred assets.

The budget-constrained Cobb-Douglas problem has a closed-form solution (no numerical optimizer needed).

> __Optimal allocation:__
>
> Partition assets into preferred $S^+ = \{i : \gamma_i > 0\}$ and non-preferred $S^- = \{i : \gamma_i \leq 0\}$ based upon the preference weights. Then, the optimal number of shares for asset $i$ is given by:
> $$n_i^{\star} = \frac{\gamma_i}{\sum_{j \in S^+} \gamma_j} \cdot \frac{B_{\text{adj}}}{P_i} \qquad \text{for } i \in S^+$$
>
> $$n_i^{\star} = \epsilon \qquad \text{for } i \in S^-$$
>
> where $B_{\text{adj}} = B - \epsilon \sum_{k \in S^-} P_k$ is the budget remaining after funding minimum positions. The budget is split among preferred assets in proportion to their preference weights, adjusted for price. The allocation responds immediately to changes in $\gamma_i$; no matrix inversion, no quadratic program, no numerical solver.

The Cobb-Douglas allocator needs preference weights $\gamma_i$ as input. Where do they come from? From a combination of **fundamental signals** (the Single Index Model) and a **sentiment signal** $\lambda_t$. There are many ways to compute the preference weights, and the sentiment signal. However, let's look at one specific formulation that captures the intuition.

> __Preference Weight Computation:__
>
> Let the **Single Index Model (SIM) parameters** for asset $i$ be given by $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$. These are estimated (at least initially) from historical data: $\alpha_i$ is the firm-specific excess growth (Jensen's alpha), $\beta_i$ is the sensitivity to the market factor (systematic risk), and $\sigma_{\varepsilon,i}$ is the residual volatility. Under the SIM, asset $i$'s expected one-step growth rate is $\mathbb{E}[g_i] = \alpha_i + \beta_i\,\mathbb{E}[g_{\mathrm{mkt}}]$. We want a preference-weight functional of the form $\gamma_i = \tanh\!\big(\text{SIM growth score for asset }i\big)$, but modulated by sentiment so high-beta names are penalized in bearish regimes and amplified in bullish ones.
>
> To build that modulation, we inflate or deflate each SIM term by a **sentiment-scaled power of $\beta_i$**: divide by $\beta_i^{\lambda_t}$, where $\lambda_t$ is the sentiment signal (defined below). When $\lambda_t > 0$ (bearish) this divisor grows with $\beta_i$ and shrinks the high-beta contribution; when $\lambda_t < 0$ (bullish) it does the reverse; at $\lambda_t = 0$ the divisor is 1 and the raw SIM score returns. Applied to both terms of the SIM growth score:
> $$\gamma_i = \tanh\!\left(\frac{\alpha_i}{\beta_i^{\lambda_t}} + \frac{\beta_i}{\beta_i^{\lambda_t}} \cdot \mathbb{E}[g_{\mathrm{mkt}}]\right)$$
>
> Simplifying the second term gives the course-wide preference-weight formula:
>
> $$
\boxed{
\gamma_i = \tanh\!\left(\frac{\alpha_i}{\beta_i^{\lambda_t}} + \beta_i^{1-\lambda_t} \cdot \mathbb{E}[g_{\mathrm{mkt}}]\right)\quad\blacksquare
}
> $$
> The factor $\beta_i^{\lambda_t}$ is the sentiment-modulated adjustment:
> * **Bearish** ($\lambda_t > 0$): high-beta assets are penalized exponentially; the denominator grows, pushing $\gamma_i$ toward zero or negative.
> * **Bullish** ($\lambda_t < 0$): high-beta assets are amplified; the risk factor shrinks, letting the growth signal dominate.
> * **Neutral** ($\lambda_t = 0$): $\beta_i^0 = 1$, so the raw SIM growth score drives preferences.
>
> The $\tanh$ activation maps the result to $(-1, 1)$, giving us well-behaved Cobb-Douglas exponents. These preference weights then flow directly into the Cobb-Douglas analytical solution to determine share allocations.

How do we compute the sentiment signal $\lambda_t$? Many approaches work. We propose a simple exponential moving average (EMA) crossover signal that captures short-term momentum as a proxy for market sentiment.

> __The Lambda Signal:__
>
> The exponential moving average (EMA) of a price series $S_t$ (USD/share) with window-size $L$ (days) is given by:
>
> $$\bar{S}_{t} = \alpha\, S_{t} + (1 - \alpha)\,\bar{S}_{t-1}, \qquad \alpha = \frac{2}{L + 1}$$
>
> For this strategy we compute two EMAs: a **short-term** ($L_{\mathrm{short}} = 21$ days) that tracks recent momentum, and a **long-term** ($L_{\mathrm{long}} = 63$ days) that tracks the underlying trend. The crossover between short and long EMAs produces the sentiment signal:
>
> $$\lambda_t = -G \cdot \left(\frac{\bar{S}^{\text{short}}_t}{\bar{S}^{\text{long}}_t} - 1\right)$$
>
> where $G > 0$ is a gain constant that controls sensitivity. The interpretation:
> * $\lambda_t > 0$: **bearish** (short EMA below long EMA). The engine becomes risk-averse.
> * $\lambda_t < 0$: **bullish** (short EMA above long EMA). The engine takes on more risk.
> * $\lambda_t \approx 0$: **neutral** (no strong signal).
>
> The EMA crossover is one choice of sentiment signal. Other sources, such as NLP-derived scores from news articles, earnings calls, or regulatory filings, can also drive $\lambda_t$.

The allocator framework is agnostic to the signal source; any scalar that maps market conditions to a risk-aversion level will work. We explore richer sentiment signals in Session 4.

> __Example__
>
> [▶ Let's build the Cobb-Douglas allocator and compare utility functions](eCornell-AI-Finance-S2-Example-BuildCobbDouglasAllocator-May-2026.ipynb). We generate a synthetic market path, compute SIM parameters and the sentiment signal, build the Cobb-Douglas utility allocator, and compare its allocation behavior to CES and log-linear alternatives.
>
> [▶ Let's replace EMA crossover with the HMM regime posterior](eCornell-AI-Finance-S2-Example-RegimeAwareSentiment-May-2026.ipynb) *(optional)*. We build a regime-aware $\lambda$ signal from the HMM posterior, run the engine with both EMA and regime-based sentiment on the same path, and compare their distributional performance across 5,000 synthetic futures.

___

### CES (Constant Elasticity of Substitution)
Cobb-Douglas is not the only choice of utility function. Different utility functions encode different assumptions about how investors trade off between assets. Here we introduce two alternatives that we compare in the examples.

> __CES Utility:__
>
> The CES utility function generalizes Cobb-Douglas by introducing an elasticity of substitution parameter $\sigma$ that controls how easily the investor substitutes between assets:
> $$U_{\text{CES}}(\mathbf{n}) = \left(\sum_{i \in S^+} \gamma_i \cdot n_i^{\rho}\right)^{1/\rho}, \qquad \rho = \frac{\sigma - 1}{\sigma}$$
>
> where $\sigma > 0$ is the **elasticity of substitution**, controlling how willing the investor is to shift allocation between assets, and $n_i$ is the number of shares of asset $i$. The CES utility reduces to Cobb-Douglas when $\sigma = 1$ (equivalently $\rho = 0$), but allows more flexible substitution patterns as $\sigma$ varies:
> * $\sigma \to 1$: CES converges to Cobb-Douglas (unit elasticity).
> * $\sigma \to \infty$: CES converges to linear utility (perfect substitutes, all-in on the best asset).
> * $\sigma \to 0$: CES converges to Leontief (perfect complements, equal allocation).
>
> **When to use CES:** When you want to control how concentrated the allocation becomes: high $\sigma$ produces more concentrated portfolios; low $\sigma$ produces more diversified ones.

Unlike Cobb-Douglas, CES has no closed-form solution, so we solve for the optimal allocation numerically. This is a trade-off: we gain flexibility in substitution patterns at the cost of analytical tractability. We also need a model for how $\sigma$ should vary with market conditions.

> __Adaptive Elasticity:__
>
> A fixed $\sigma$ ignores the market environment. We can do better by linking $\sigma$ to the same sentiment signal $\lambda_t$ that drives the preference weights:
> $$\boxed{\sigma(\lambda_t) = \sigma_{\min} + \frac{\sigma_{\max} - \sigma_{\min}}{1 + |\lambda_t|}\quad\blacksquare}$$
>
> The interpretation:
> * **Neutral** ($\lambda_t \approx 0$): $\sigma \to \sigma_{\max}$. The allocator concentrates budget in the highest-scoring asset, because the sentiment signal carries no warning.
> * **Extreme sentiment** ($|\lambda_t|$ large): $\sigma \to \sigma_{\min}$. The allocator diversifies, hedging against the possibility that the strong signal is wrong.
>
> The $|\lambda_t|$ in the denominator makes the response symmetric: both strongly bullish and strongly bearish sentiment trigger diversification. This is intentional: extreme conviction in either direction increases the risk that the signal is noise.
>
> **Default bounds:** $\sigma_{\min} = 0.5$ and $\sigma_{\max} = 5.0$. Below $\sigma \approx 0.3$ the CES demand function flattens and $\gamma_i$ differences barely affect allocation (the allocator becomes unresponsive to preferences). Above $\sigma \approx 5$ the allocator effectively picks the single highest-$\gamma$ asset, making the portfolio a one-stock bet. The $(0.5, 5.0)$ bounds keep the allocator preference-responsive on one end and concentration-capped on the other.

The adaptive elasticity turns $\sigma$ from a static hyperparameter into a second adaptive channel. The preference weights $\gamma_i$ decide _which_ assets to favor; the elasticity $\sigma(\lambda_t)$ decides _how aggressively_ to concentrate on them.


### Log-Linear Utility

> __Log-Linear Utility:__
>
> Log-linear utility is the logarithmic transform of Cobb-Douglas. It produces the **same optimal allocation** (the log transform preserves the optimum), but the utility _values_ differ. This matters when utility is used as a reward signal, for example in the bandit algorithms we introduce in Session 3.
> $$U_{\text{log}}(\mathbf{n}) = \sum_{i \in S^+} \gamma_i \cdot \log(n_i)$$
>
> **When to use log-linear:** When you want the same allocation as Cobb-Douglas but need utility values that are additive (useful for averaging across scenarios or feeding into learning algorithms).

___

## The Rebalancing Engine: Pseudo-code

The utility allocator gives us a principled way to compute positions at any point in time. The rebalancing engine wraps this allocator in a daily loop with safety rules:

> __Rebalancing Engine:__
>
> The engine operates in three phases: one-time initialization, a daily loop, and a history-writing output.
>
> **Phase 1: Initialization**
> 1. Load market price path $S_t$ for a broad index.
> 2. Compute short-term EMA ($L_{\mathrm{short}} = 21$) and long-term EMA ($L_{\mathrm{long}} = 63$). Define warmup offset $t_0 = L_{\mathrm{short}} + L_{\mathrm{long}}$.
> 3. Build the sentiment series: $\lambda_t = -G \cdot (\bar{S}^{\text{short}}_t / \bar{S}^{\text{long}}_t - 1)$.
> 4. Smooth market growth into $g_{m,t}$ using an EMA of the log growth rates ($L_{\mathrm{growth}} = 10$).
> 5. Assemble the **context model**: budget $B$, ticker universe, price matrix, SIM parameters, risk floor $\epsilon$, and market factor $g_{m,t}$.
> 6. Define **trigger rules**: drawdown limit, turnover cap, and reallocation schedule.
> 7. Compute initial allocation via Cobb-Douglas utility maximization.
>
> **Phase 2: Daily Loop** (for $t = t_0 + 1, \ldots, t_0 + T$)
> 1. Read reallocation flag $a_t$ from schedule.
> 2. **If $a_t = 1$ (rebalance day)**, liquidate current positions at today's prices, check the drawdown trigger (if drawdown exceeds $d_{\max}$, de-risk to 100% cash; otherwise update $\lambda_t$, compute new $\gamma_i$ preferences, and solve the Cobb-Douglas allocation), then check the turnover cap (if proposed trades exceed $\tau_{\max}$, scale down proportionally).
> 3. **If $a_t = 0$ (hold day)**, propagate prior allocation unchanged.
>
> **Phase 3: Output**
> Return history indexed by trading day: positions, preference weights, cash, and total wealth.
>
> The output history is what every scorecard and diagnostic example in this session consumes.

The following examples implement this engine and compare it against the Session 1 baselines.

> __Example__
>
> [▶ Let's wire the allocator into a rebalancing engine and produce a scorecard](eCornell-AI-Finance-S2-Example-RebalancingEngineScorecard-May-2026.ipynb). We run the Cobb-Douglas engine with trigger rules on a synthetic market path, compare it head-to-head against the Session 1 baselines, and sweep the CES elasticity parameter to see how concentration tunes engine behavior.
>
> [▶ Let's compare static and adaptive allocation across 5,000 paths](eCornell-AI-Finance-S2-Example-StaticVsAdaptiveComparison-May-2026.ipynb). We run both strategies on the full Monte Carlo ensemble, measure how often the engine wins, identify which regimes drive the edge, and compare paired quantiles across the wealth distribution.

___

## Trigger Rules: The Safety Net

An adaptive engine that trades freely is dangerous; it can churn the portfolio, chase noise, or fail to protect capital during a crash. Trigger rules are the _guardrails_ that keep the engine safe:

| Rule | Parameter | What It Does |
|------|-----------|-------------|
| **Drawdown Limit** | $d_{\max}$ (e.g., 10%) | If portfolio wealth drops more than $d_{\max}$ from its peak, the engine de-risks to 100% cash. This is a circuit breaker. |
| **Turnover Cap** | $\tau_{\max}$ (e.g., 50%) | If the proposed rebalance would trade more than $\tau_{\max}$ of portfolio value, the trades are scaled down proportionally. This controls transaction costs. |
| **Reallocation Schedule** | $a_t \in \{0, 1\}$ | The binary schedule determines _which days_ the engine is allowed to rebalance. Daily, weekly, or event-driven. |

> __Why are trigger rules essential?__
>
> Without them, the engine is unconstrained: it could rebalance every day (expensive), ignore a crash (catastrophic), or flip positions  based on noise in the $\lambda$ signal. Trigger rules encode _human judgment_ about acceptable risk into the machine's decision loop. They are the bridge between autonomous operation and investment committee oversight.

In Session 4, we extend these rules with human override protocols and escalation procedures for production deployment.

> __Example__
>
> [▶ Let's stress-test the rebalancing engine on 5,000 synthetic futures](eCornell-AI-Finance-S2-Example-StressTestRebalancingEngine-May-2026.ipynb). We generate the full Monte Carlo scenario ensemble, run all strategies, and compute the tail-risk scorecard (VaR, CVaR, drawdown, NPV) to evaluate how the engine performs under stress.
>
> [▶ Let's diagnose turnover costs and attribution](eCornell-AI-Finance-S2-Example-TurnoverAttributionDiagnostics-May-2026.ipynb) *(optional)*. We measure transaction cost sensitivity, decompose performance into a 3-way attribution (utility choice, dynamic reallocation, trigger rules), and visualize the rebalance event timeline.

___

## Summary

In this session, we replaced the static minimum-variance allocation from Session 1 with an adaptive rebalancing engine that reads market conditions and adjusts the portfolio daily within explicit safety constraints.

> __Key Takeaways:__
>
> * __Utility maximization replaces variance minimization:__ The Cobb-Douglas utility function allocates budget proportionally to preference weights via a closed-form analytical solution. CES and log-linear alternatives offer different concentration and diversification trade-offs using the same preference inputs.
> * __Preferences adapt dynamically to market conditions:__ Preference weights combine SIM fundamentals with a sentiment signal derived from EMA crossover, so the allocator shifts risk appetite as conditions change. When sentiment turns bearish, high-beta assets are penalized; when bullish, they are amplified.
> * __Trigger rules encode human judgment:__ Drawdown limits, turnover caps, and reallocation schedules act as safety guardrails that constrain the engine's behavior. These rules ensure the engine operates within bounds that a human investment committee would approve.

These three takeaways map directly to the three claims Amara presents in the [Session 2 introduction](eCornell-AI-Finance-S2-Introduction-MayasAdaptiveEngine-May-2026.ipynb). With the utility function, the sentiment-driven preferences, and the trigger rules now operational, Maya can walk into Thursday's risk committee meeting with the fifth-percentile bear-subset wealth figure that the [stress-test](eCornell-AI-Finance-S2-Example-StressTestRebalancingEngine-May-2026.ipynb) and [static-vs-adaptive](eCornell-AI-Finance-S2-Example-StaticVsAdaptiveComparison-May-2026.ipynb) examples produce.

__What comes next?__ In [Session 3](../session-3/eCornell-AI-Finance-S3-Lecture-OnlineLearningValidation-May-2026.ipynb), we teach the engine to learn: EWLS updates SIM parameters online as new data arrives, a multi-armed bandit discovers the best CES elasticity per regime, and a formal validation report gates deployment.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The rebalancing engine described here is a pedagogical tool using synthetic data and simplified models. Real-world algorithmic trading involves regulatory requirements, execution risk, and operational considerations beyond the scope of this session.

___